# Decision Engine — CSV Offline Runner

Loads vw3–vw6 from exported CSVs and runs the full decision engine pipeline
(QA gates → Layers 2–4 → Layers 5–8) without a live Presto connection.

**Prerequisites:** Export vw3–vw6 from DBeaver/Presto and place CSVs in `../data/csv_exports/`.

In [ ]:
import sys, pathlib, logging

ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s %(message)s')

import pandas as pd
import numpy as np

from telecom_credit_engine.orchestration.orchestrator import (
    run_sql_qa_gates,
    run_decision_engine_step,
    run_layers_5_to_8_step,
)
from telecom_credit_engine.decisioning.run_credit_v1_decision_engine import (
    DECISION_ENGINE_CONFIG,
    ACTION_SPECS_DF,
)
from telecom_credit_engine.state_management.run_layers_5_to_8 import (
    STATE_ENGINE_CONFIG,
    INTERVENTION_CONFIG,
    PORTFOLIO_CONFIG,
    GOVERNANCE_CONFIG,
)

CSV_DIR = ROOT / 'data' / 'csv_exports'
OUT_DIR = CSV_DIR / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Imports OK')

## Load CSVs

In [ ]:
DATE_COLS_VW3 = ['feature_dt', 'last_disbursement_dt', 'last_repayment_dt']
DATE_COLS_VW4 = ['feature_dt']
DATE_COLS_VW5 = ['feature_dt']
DATE_COLS_VW6 = ['feature_dt']

layer1_df = pd.read_csv(CSV_DIR / 'vw3_layer1_features.csv', parse_dates=DATE_COLS_VW3)
layer0_df = pd.read_csv(CSV_DIR / 'vw4_layer0_scores.csv',   parse_dates=DATE_COLS_VW4)
vw5_df    = pd.read_csv(CSV_DIR / 'vw5_reason_codes.csv',     parse_dates=DATE_COLS_VW5)
vw6_df    = pd.read_csv(CSV_DIR / 'vw6_cap_and_action.csv',   parse_dates=DATE_COLS_VW6)

for df in [layer1_df, layer0_df, vw5_df, vw6_df]:
    if 'run_date' in df.columns:
        df.drop(columns=['run_date'], inplace=True)

print(f'layer1_df: {layer1_df.shape}')
print(f'layer0_df: {layer0_df.shape}')
print(f'vw5_df:    {vw5_df.shape}')
print(f'vw6_df:    {vw6_df.shape}')

## Type Coercion

In [ ]:
SKIP_COLS = {'feature_dt', 'subscriber_msisdn', 'last_disbursement_dt',
             'last_repayment_dt', 'primary_reason_code', 'recommended_action'}

def coerce_numerics(df):
    for col in df.columns:
        if col in SKIP_COLS:
            continue
        if df[col].dtype == object:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

INT_PATTERNS = ('_flag', '_cnt_', '_days_', '_day', 'wallet_active_days')

def coerce_int_flags(df):
    for col in df.columns:
        if any(p in col for p in INT_PATTERNS) and pd.api.types.is_float_dtype(df[col]):
            df[col] = df[col].fillna(0).astype(int)
    return df

layer1_df = coerce_int_flags(coerce_numerics(layer1_df))
layer0_df = coerce_numerics(layer0_df)
vw5_df    = coerce_numerics(vw5_df)
vw6_df    = coerce_numerics(vw6_df)

print('Type coercion complete')
print(layer1_df.dtypes)

## Data Inspection

In [ ]:
for name, df in [('layer1', layer1_df), ('layer0', layer0_df),
                  ('vw5', vw5_df), ('vw6', vw6_df)]:
    print(f'\n=== {name} ===')
    print(f'Shape: {df.shape}')
    print(f'Nulls:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
    display(df.head(3))

In [ ]:
layer1_df.describe()

In [ ]:
layer0_df.describe()

## QA Gates

In [ ]:
run_sql_qa_gates(layer1_df, layer0_df, vw5_df, vw6_df)
print('QA gates passed')

## Run Decision Engine (Layers 2–4)

In [ ]:
engine_outputs = run_decision_engine_step(
    layer1_df, layer0_df, vw5_df, vw6_df,
    DECISION_ENGINE_CONFIG, ACTION_SPECS_DF,
)

for key, val in engine_outputs.items():
    shape = val.shape if isinstance(val, pd.DataFrame) else type(val)
    print(f'{key}: {shape}')

## Inspect Decision Engine Outputs

In [ ]:
vw9 = engine_outputs['vw9_credit_v1_final_capacity_output']
print('=== vw9 shape:', vw9.shape)
print('\n=== Action distribution ===')
print(vw9['selected_action'].value_counts())
print('\n=== Credit limit stats ===')
print(vw9['CreditLimit'].describe())

In [ ]:
vw7 = engine_outputs['vw7_credit_v1_policy_prefilter']
print('=== vw7 shape:', vw7.shape)
display(vw7.head())

vw8 = engine_outputs['vw8_credit_v1_tnv_action_evaluation']
print('\n=== vw8 shape:', vw8.shape)
display(vw8.head())

In [ ]:
# Spot-check: SEVERE_DSI subscribers should be RESTRICT with limit=0
severe = vw6_df[vw6_df['primary_reason_code'] == 'SEVERE_DSI'] if 'primary_reason_code' in vw6_df.columns else vw5_df[vw5_df['primary_reason_code'] == 'SEVERE_DSI']
if len(severe) > 0:
    severe_msisdns = severe['subscriber_msisdn'].unique()
    check = vw9[vw9['subscriber_msisdn'].isin(severe_msisdns)]
    print(f'SEVERE_DSI subscribers: {len(severe_msisdns)}')
    print(f'  All RESTRICT: {(check["selected_action"] == "RESTRICT").all()}')
    print(f'  All limit=0:  {(check["CreditLimit"] == 0).all()}')
else:
    print('No SEVERE_DSI subscribers found in reason codes')

## Run Layers 5–8 (State, Intervention, Portfolio, Governance)

In [ ]:
layers_output = run_layers_5_to_8_step(
    engine_outputs,
    STATE_ENGINE_CONFIG,
    INTERVENTION_CONFIG,
    PORTFOLIO_CONFIG,
    GOVERNANCE_CONFIG,
)

print(f'circuit_breaker_multiplier: {layers_output["circuit_breaker_capacity_multiplier"]}')
print(f'portfolio_control_signal:   {layers_output["portfolio_control_signal"]}')

## Inspect Layers 5–8 Outputs

In [ ]:
state_df = layers_output['state_df']
print('=== State distribution ===')
print(state_df['operating_state'].value_counts())
print(f'\nTotal subscribers: {len(state_df)}')
display(state_df.head())

In [ ]:
intervention_df = layers_output['intervention_df']
print('=== Intervention DF ===')
print(f'Shape: {intervention_df.shape}')
display(intervention_df.head())

portfolio_df = layers_output['portfolio_monitor_df']
print('\n=== Portfolio Monitor DF ===')
display(portfolio_df)

governance_df = layers_output['governance_log_df']
print('\n=== Governance Log DF ===')
print(f'Shape: {governance_df.shape}')
display(governance_df.head())

## Save Outputs to CSV

In [ ]:
engine_outputs['vw7_credit_v1_policy_prefilter'].to_csv(OUT_DIR / 'vw7_policy_prefilter.csv', index=False)
engine_outputs['vw8_credit_v1_tnv_action_evaluation'].to_csv(OUT_DIR / 'vw8_tnv_evaluation.csv', index=False)
engine_outputs['vw9_credit_v1_final_capacity_output'].to_csv(OUT_DIR / 'vw9_final_capacity_output.csv', index=False)

layers_output['state_df'].to_csv(OUT_DIR / 'state_df.csv', index=False)
layers_output['portfolio_monitor_df'].to_csv(OUT_DIR / 'portfolio_monitor_df.csv', index=False)

print(f'Outputs saved to {OUT_DIR}')
for f in sorted(OUT_DIR.glob('*.csv')):
    print(f'  {f.name} — {f.stat().st_size / 1e6:.1f} MB')